# Temporal analysis

Temporal methods address different properties of a time-bearing table.
`dates` compares observed calendar days with an inferred cadence;
`sortedness` compares column values with existing row order; `breaks`
looks for changes within a series ordered by a selected time axis.
Declared [expectations](expectations.ipynb) supply a calendar, ordering,
or panel reference when the intended structure is known.

The inferred date-gap analysis needs at least 30 distinct days. Other
passes have their own eligibility rules and sample sizes.


## Modal cadence and gaps

The pass sorts and deduplicates the observed days, then measures the gaps
between consecutive distinct days. The modal gap (most frequent) is the
column's assumed cadence; the median gap is reported alongside it. Gap
semantics depend on the cadence:

- **daily cadence** (modal 1): missing *weekdays* are counted, and a pure
  weekend bridge is not a gap. Holidays are not modeled — the finding's
  reference says so explicitly (`weekdays`, source `inferred`).
- **weekly cadence** (modal 7): gaps are counted in weeks once more than
  ten days elapse.
- **other cadences**: a gap is flagged when the elapsed time exceeds
  1.5× the modal interval *and* the modal interval plus two days; the
  count is expressed in modal-interval units and marked approximate.

At most 10 gap events are reported, ordered by missing count. Each
`gaps` finding carries its reference — the cadence or weekday rule that
defines "missing" — because an empty interval is only meaningful against
an expectation.

## Irregular intervals

The coefficient of variation (CV) of the day gaps measures variation in
observed spacing. A CV above 0.5, with at least ten gaps, produces an
`intervals` statement containing the modal interval, median interval, and
CV. Irregular spacing can affect resampling and lag features that assume
a regular time index.


## Sortedness and late backfills

With at least 100 non-null values, `sortedness` reports strictly increasing
or non-decreasing order when it holds. If the column is not sorted, it also
checks a narrower backfill pattern: at most three descents, at least three
values after the first descent, and no tail value above the preceding
peak. The evidence records the seam and tail size. This can identify a
late block appended out of order; it is not a general unsortedness alarm.

Use `hints={"order_by": [...]}` to check a required row order and report
violations against that declaration.


In [ ]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(51)

# Frame A: a daily snapshot calendar with a two-weekday hole.
base = dt.date(2024, 1, 1)
skip = {dt.date(2024, 2, 8), dt.date(2024, 2, 9)}
business, i = [], 0
while len(business) < 3_000:
    d = base + dt.timedelta(days=i // 10)
    if d not in skip:
        business.append(d)
    i += 1
daily = pl.DataFrame({"row_id": range(len(business)), "snapshot_date": business})

report = onceover.profile(daily)
for f in report.findings:
    if f["category"] in {"gaps", "ordering"}:
        m = f["evidence"][0]["metrics"]
        print(f"[{f['category']:9s}] {f['claim']}")
        if f["category"] == "gaps":
            print(f"            reference={m.get('reference')} cadence_days={m.get('cadence_days')}")

# Frame B: a bursty event log — an uneven clock with silences.
events, day = [], 0
while len(events) < 3_000:
    for _ in range(5):                # five consecutive active days
        day += 1
        events.append(day)
    day += rng.randrange(9, 15)       # then a silence
log_dates = [dt.date(2024, 1, 1) + dt.timedelta(days=d) for d in events]
rng.shuffle(log_dates)                # logs arrive out of order
log = pl.DataFrame({"event_id": range(len(log_dates)), "event_day": log_dates})

log_report = onceover.profile(log)
log_gaps = [f for f in log_report.findings if f["category"] == "gaps"]
print("gap events in the event log:", len(log_gaps))
for f in log_gaps[:2]:
    print(f"[gaps      ] {f['claim']}")
for f in log_report.findings:
    if f["category"] == "intervals":
        print(f"[intervals ] {f['claim']}")

The snapshot calendar is missing two weekdays under the inferred weekday
reference. Its dates are non-decreasing in row order. The event log has
irregular intervals: gaps are one day within bursts and ten or more days
between bursts. Those longer intervals also produce gap events against
the daily modal interval. The report retains the ten largest gap events;
the example prints two of them. The shuffled event-log date column
receives no sortedness finding.

## Datetime columns

Datetime analysis uses day granularity. It does not detect time-of-day
patterns, intraday gaps, or session boundaries
([](../appendix/directions.ipynb)). Gap counts use all distinct days, but
their interpretation depends on the inferred cadence or weekday reference.


## Sustained changes in an ordered series

The `breaks` pass chooses a time axis in this order: a usable date,
datetime, or integer column declared by `hints["time"]`; the first date or
datetime column; then a year-like integer column with values in 1900–2100
and at least eight distinct values. Rows are sorted by that axis and
sampled to at most 10,000 at standard or 50,000 at deep. Numeric series
are limited to 16 or 32, respectively. Coverage records the selected axis
and the amount of work completed.

Candidate breakpoints are fitted on a deterministic training partition.
The fixed segmentation and fitted parameters are then evaluated on
holdout rows. The three sustained change types compare different models:

- `level_shift` splits a constant-mean reference into segment means,
  recursively finding up to three level breaks at standard or four at
  deep. Candidates need at least 30 observations per training segment.
- `trend_change` compares one fitted line with two lines on either side
  of the selected breakpoint and records their slopes.
- `variance_change` proposes a split using squared deviations from the
  global mean, then records separate segment variances.

Candidates need at least 10% training improvement. Reporting also checks
holdout improvement (15% by default) under the relevant error or likelihood
comparison. These are selected model comparisons, not p-values. When a
pooled level change is absent, a bounded follow-up can inspect levels of
up to two eligible text grouping columns, with at most eight levels each.

The [residual-shift pass](local_linear_regimes.ipynb) instead measures a
change after accounting for other predictors. [Period associations](period_associations.ipynb)
compare relationships between two columns. [Model-routing drift](drift.ipynb)
compares later predictor distributions with an early-period model. Those
findings should not be interpreted as interchangeable time tests.


## Transient excursions

The breaks pass looks for change inside a single time-ordered series.
Its sustained types — `level_shift`, `trend_change`, `variance_change`
— describe a series that moves and stays moved. The same pass also
scans for the opposite shape: an *excursion*, a stretch of points that
departs from the local baseline and returns to it — a one-day demand
collapse, a three-hour latency failure. Excursions are reported as
`changes` findings of type `transient_excursion` (form `excursion`),
with the event's start, end, and peak in the evidence.

The baseline is the series' own recent past. When the median interval
between timestamps divides a week evenly and the series spans at least
two and a half weeks of periods, the baseline for a point is the
median of the values at the same time of week in the three
neighbouring weeks on each side — matched by timestamp, not by
position, so a missing or sampled-out grid point does not shift the
phase. Otherwise the baseline is a centered rolling median over 97
points. Deviations are scaled by the median absolute residual
(× 1.4826); points beyond 8 robust standard units are grouped into
events, bridging short unflagged gaps. An event's span is capped — one
period on the seasonal baseline, half the window on the rolling one —
so a sustained level shift, whose run of deviating points exceeds the
cap, is left to the level-shift scan rather than reported as a very
long excursion. An event at the end of the series counts: no return
to baseline is required. At most ten events per series are reported,
ranked by peak deviation.

The example plants a single one-day collapse in eight weeks of hourly
requests with a weekly pattern. The scan reports that day, at a peak
of −13.2 robust units, and nothing else.

In [ ]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(3)
start = dt.datetime(2026, 1, 5, 0, 0)
ts, vals = [], []
for h in range(24 * 7 * 8):
    t = start + dt.timedelta(hours=h)
    weekly = 100 + 30 * (1 if t.weekday() < 5 else 0.4)
    v = weekly + rng.gauss(0, 4)
    if dt.datetime(2026, 2, 11) <= t < dt.datetime(2026, 2, 12):
        v -= 55                      # the planted one-day collapse
    ts.append(t)
    vals.append(round(v, 2))

report = onceover.profile(pl.DataFrame({"ts": ts, "requests": vals}))
for f in report.findings:
    if f.get("type") == "transient_excursion":
        print(f["claim"])